# Notebook 06 — Engine Backtest & Performance Tearsheet

Demonstrates the **engine path**: `TrendSignal → BacktestEngine + TrendSignalStrategy → BacktestResult → PerformanceCharts`.

Unlike `BarBacktest` (which simulates each symbol independently), `BacktestEngine` runs a full
portfolio rebalancing loop — tracking NAV in dollars, logging every trade, and recording
portfolio weights over time.

| Component | Role |
|---|---|
| `TrendSignal` | Generates `signal_open` from MA-200 crossover |
| `TrendSignalStrategy` | Translates signal into equal-weight targets at each rebalance bar |
| `BacktestEngine` | Iterates bars, calls strategy, rebalances portfolio, tracks NAV |
| `BacktestResult` | NAV series, daily returns, weights DataFrame, trade log |
| `PerformanceCharts` | Interactive Plotly tearsheet, equity curve, weights chart |

In [1]:
import pandas as pd

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest import BacktestEngine, TrendSignalStrategy
from hailmary.viz.performance_charts import PerformanceCharts

## 1. Universe & Dates

In [6]:
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"] # "ADA-USD", "DOT-USD", "AVAX-USD", "LINK-USD", "MATIC-USD"]

start = pd.Timestamp("2022-01-01")
end   = pd.Timestamp("2024-12-31")

print(f"Universe : {symbols}")
print(f"Period   : {start.date()} → {end.date()}")

Universe : ['BTC-USD', 'ETH-USD', 'SOL-USD']
Period   : 2022-01-01 → 2024-12-31


## 2. Fetch Data with Warmup

The MA-200 needs 200 bars of history before it is valid.  We fetch `signal.warmup` extra
business days before `start` so the signal is fully populated from day one of the backtest.

In [7]:
signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()

fetch_start = start - pd.offsets.BDay(signal.warmup)
bars = yahoo.get_bars(symbols, start=fetch_start, end=end)

n_sym   = bars.index.get_level_values("symbol").nunique()
ts_min  = bars.index.get_level_values("timestamp").min().date()
ts_max  = bars.index.get_level_values("timestamp").max().date()
print(f"Fetched {len(bars):,} bars across {n_sym} symbols  ({ts_min} → {ts_max})")

2026-04-26 17:23:36.994 | INFO     | hailmary.data.providers.yahoo:get_bars:64 - Fetching 3 symbols from Yahoo (2021-03-29 00:00:00 → 2024-12-31 00:00:00); inclusive
2026-04-26 17:23:37.126 | DEBUG    | hailmary.data.cache:set:45 - Cached 4122 rows key=8c0b2d56b6b0


Fetched 4,122 bars across 3 symbols  (2021-03-29 → 2024-12-31)


## 3. Generate Signal

In [12]:
signal_df = signal.run(bars, trim_start=start)

ts_min = signal_df.index.get_level_values("timestamp").min().date()
ts_max = signal_df.index.get_level_values("timestamp").max().date()
print(f"Signal period : {ts_min} → {ts_max}")
print(f"MA NaN count  : {signal_df['ma'].isna().sum()}  (should be 0)")
print(f"signal_df columns  : {signal_df.columns}")

# Days in signal per symbol
signal_df["signal_open"].unstack("symbol").sum().rename("days_in_signal").to_frame()

Signal period : 2022-01-01 → 2024-12-31
MA NaN count  : 0  (should be 0)
signal_df columns  : Index(['open', 'high', 'low', 'close', 'volume', 'ma', 'signal_close',
       'signal_open', 'trade_direction', 'turnover', 'enter', 'exit', 'cycle',
       'signal_age'],
      dtype='str')


,days_in_signal
symbol,
BTC-USD,586
ETH-USD,549
SOL-USD,512


## 4. Run BacktestEngine

`TrendSignalStrategy` reads `signal_open` at each bar and sets equal weights across all
symbols currently above their MA-200.  Each in-signal symbol gets `1 / N_universe` weight;
uninvested capital sits as cash.

The engine checks signals on every trading day and only executes trades when `signal_open`
actually changes — no unnecessary rebalancing between flips.

`fill_mode="conservative"` fills buys at the bar high and sells at the bar low on signal
entry and exit days, giving a worst-case execution cost on every trade.

In [ ]:
strategy = TrendSignalStrategy(signal_df)

engine = BacktestEngine(
    strategy,
    bars=bars,
    initial_capital=1_000_000,
    fill_mode="conservative",
)
result = engine.run()

print(f"Total trades: {len(result.trade_log)}")

## 5. BacktestResult Overview

In [ ]:
result.summary()

In [ ]:
print(f"NAV start : ${result.nav.iloc[0]:,.0f}")
print(f"NAV end   : ${result.nav.iloc[-1]:,.0f}")

In [ ]:
result.trade_log.head(10)

## 6. Performance Tearsheet

4-panel chart: equity curve, drawdown, rolling 252-day Sharpe, return distribution.

In [ ]:
charts = PerformanceCharts(result, name="MA-200 Crypto Trend")
charts.tearsheet().show()

## 7. Portfolio Weights Over Time

Stacked area chart showing which assets are held at each point in time.
Empty gaps indicate periods where no symbol was above its MA-200 (fully in cash).

In [ ]:
charts.weights_chart().show()

## 8. Monthly Returns Heatmap

In [ ]:
charts.monthly_returns_heatmap().show()

## 9. Return Distribution

In [ ]:
charts.returns_distribution().show()